# System Modeling


## 실행 준비

<!-- exercise-guidance:control-2/system-modeling/execution -->
Python 경로는 python-control을 사용합니다. 모듈이 없으면 Colab·Jupyter의 코드 셀에서 `%pip install control==0.10.2`를 실행하세요.
<!-- /exercise-guidance:control-2/system-modeling/execution -->


## 학습 목표

<!-- exercise-guidance:control-2/system-modeling/objective -->
질량이나 저항을 바꿀 때 Transient Response와 Steady-State Value 중 무엇이 변하는지 모델로 판단합니다.
<!-- /exercise-guidance:control-2/system-modeling/objective -->


## Cruise 모델과 단위

<!-- exercise-guidance:control-2/system-modeling/activity-1 -->
### 시작 조건

$m=1000$ kg, $b=50$ N·s/m, 힘 500 N. State와 Output은 Velocity [m/s]입니다.

### 진행 방법

1. $A=-b/m$, $B=1/m$, $C=1$, $D=0$으로 모델을 만들고 500 N의 일정한 힘에 대한 $0\le t\le150$ s 응답을 그리세요.
2. $m\in\{500,1000,1500\}$만 바꾸고 같은 힘을 입력하세요. $m\dot v+bv=500$에 $\dot v=0$을 대입해 Steady-State Velocity를 구하고, $\dot v+(b/m)v=500/m$의 계수를 1차 표준형과 대응시켜 Time Constant를 구하세요.
3. 질량별로 계산한 Steady-State Velocity와 시뮬레이션 마지막 표본 $v(150)$를 나란히 기록하세요. 곡선이 계산한 Steady-State Velocity의 약 63.2%에 도달하는 시각을 찾아, 계산한 Time Constant와 대조하세요.

### 남길 결과

- 질량 3조건 표, Velocity 곡선 1개, 질량이 바꾼 특성 짧은 설명.
<!-- /exercise-guidance:control-2/system-modeling/activity-1 -->


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import control as ctl
s = ctl.tf('s')
m=1000.; b=50.; force=500.
G=ctl.ss([[-b/m]],[[1/m]],[[1]],[[0]])
t=np.linspace(0,150,1501)
t,v=ctl.step_response(force*G,t)
plt.plot(t,v); plt.xlabel('Time [s]'); plt.ylabel('Speed [m/s]'); plt.grid()
# TODO: 질량 3조건과 식으로 예상한 값을 비교하세요.


## 결과 기록·자가 점검

<!-- exercise-guidance:control-2/system-modeling/activity-1-record -->
**m [kg]:**

**예상 최종 속도 [m/s]:**

**측정 속도 [m/s]:**

**Time Constant [s]:**

### 확인할 점

- 입력 힘 500을 중복 곱하지 않았는가?
- 같은 b에서 질량 변화가 Steady-State Velocity보다 Time Constant에 영향을 주는가?
<!-- /exercise-guidance:control-2/system-modeling/activity-1-record -->


## Motor 모델의 계수 비교

<!-- exercise-guidance:control-2/system-modeling/activity-2 -->
### 시작 조건

$J=.01$, $b=.1$, $K=.01$, $L=.5$. $R\in\{.5,1,2\}$ Ω를 비교하며 입력은 1 V입니다.

### 진행 방법

1. $G(s)=\frac K{(Js+b)(Ls+R)+K^2}$를 만들고 $0\le t\le 10$ s 응답을 구하세요.
2. 각 R에서 Pole·DC Gain·최종 출력값을 표에 적으세요.
3. 전기 저항과 관성 계수가 식의 어느 항을 바꾸는지 설명하세요.

### 남길 결과

- 응답 3곡선과 계수 표 1개.
<!-- /exercise-guidance:control-2/system-modeling/activity-2 -->


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import control as ctl
s = ctl.tf('s')
J=.01; b=.1; K=.01; L=.5
for R in [.5,1.,2.]:
    G=K/((J*s+b)*(L*s+R)+K**2)
    print(R,ctl.poles(G),ctl.dcgain(G))
# TODO: 같은 시간축의 응답을 추가하세요.


## 결과 기록·자가 점검

<!-- exercise-guidance:control-2/system-modeling/activity-2-record -->
**R [Ω]:**

**Pole:**

**DC Gain:**

**최종 속도 [rad/s]:**

### 확인할 점

- 모든 조건에서 같은 J·b·K·L·입력을 사용했는가?
- 장기 출력과 1 V×DC Gain을 비교했는가?
<!-- /exercise-guidance:control-2/system-modeling/activity-2-record -->


## 측정한 응답에서 모델 추정하기

<!-- exercise-guidance:control-2/system-modeling/activity-3 -->
### 시작 조건

센서의 합성 Step 기록 $y=1.8(1-e^{-t/3})+\text{잡음}$입니다. $0\le t\le 15$ s, 151점, 잡음 표준편차 .03, `seed=7`입니다.

### 진행 방법

1. 제공된 측정 배열을 `curve_fit`의 목표값으로 사용하고, $K(1-e^{-t/\tau})$의 $K,\tau$를 추정하세요.
2. 전체 $0$–$15$ s와 앞 $0$–$3$ s의 표본에 각각 같은 추정을 적용하세요. 구간별 사용 표본 수·추정한 $K,\tau$를 표로 적으세요.
3. 각 추정 곡선을 측정값과 같은 시간축에 그리고, 잔차는 측정값−추정값으로 그리세요. 곡선의 최종값과 잔차 형태를 근거로 더 측정하고 싶은 시간 구간 하나를 정하고 이유를 적으세요.

### 남길 결과

- 두 측정 구간의 추정값 표와 응답·잔차 그림.
<!-- /exercise-guidance:control-2/system-modeling/activity-3 -->


In [ ]:
from scipy.optimize import curve_fit
rng=np.random.default_rng(7); tm=np.linspace(0,15,151)
measured=1.8*(1-np.exp(-tm/3))+rng.normal(0,.03,len(tm))
def response(t,gain,tau):return gain*(1-np.exp(-t/tau))
for end in [3,15]:
    select=tm<=end
    parameters,_=curve_fit(response,tm[select],measured[select],p0=[1,2],bounds=(0,np.inf))
    print(end,parameters)
# TODO: 곡선과 잔차를 같은 시간축으로 비교하세요.


## 결과 기록·자가 점검

<!-- exercise-guidance:control-2/system-modeling/activity-3-record -->
**기록 구간 [s]:**

**추정 Gain:**

**추정 Time Constant [s]:**

**RMSE:**

### 확인할 점

- 실제 측정처럼 합성 자료의 설정값을 모른다고 가정하고 추정했는가?
- 기록이 짧을 때의 불확실성을 잔차 크기만으로 판단하지 않았는가?
<!-- /exercise-guidance:control-2/system-modeling/activity-3-record -->


## 공식 예제·데이터

<!-- exercise-guidance:control-2/system-modeling/resources -->
- [SciPy · curve_fit](https://docs.scipy.org/doc/scipy/reference/generated/scipy.optimize.curve_fit.html)
<!-- /exercise-guidance:control-2/system-modeling/resources -->
